In [ ]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd()
print("Current working directory:", cwd)

candidates = [cwd] + list(cwd.parents)

root = None
for p in candidates:
    if (p / "data" / "raw" / "test_part_000.parquet").exists() and (p / "data" / "raw" / "test_part_003.parquet").exists():
        root = p
        break

if root is None:
    raise FileNotFoundError(
        "Cannot find data/raw/test_part_000.parquet and test_part_003.parquet in current folder or any parent folders.\n"
        "Please make sure VS Code opened the project root folder (the folder that contains 'data')."
    )

print("Project root found:", root)

P000 = root / "data" / "raw" / "test_part_000.parquet"
P003 = root / "data" / "raw" / "test_part_003.parquet"

df0 = pd.read_parquet(P000)
df3 = pd.read_parquet(P003)

print("part_000 shape:", df0.shape)
print("part_003 shape:", df3.shape)
print("columns:", df0.columns.tolist())


Current working directory: c:\Users\14176\Desktop\Honours Project\Hons Project\notebooks
Project root found: c:\Users\14176\Desktop\Honours Project\Hons Project
part_000 shape: (2830, 8)
part_003 shape: (2830, 8)
columns: ['sha256', 'url', 'label', 'target', 'date', 'lang', 'lang_score', 'html']


In [ ]:
import re
import csv
import pandas as pd
from pathlib import Path

RANDOM_STATE = 42
N_PER_CLASS = 1000
HTML_MIN_LEN = 200 

OUT_DIR = root / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def normalize_label(x):
    if pd.isna(x):
        return None
    if isinstance(x, (int, float)):
        if x == 1:
            return "phish"
        if x == 0 or x == -1:
            return "benign"
    s = str(x).strip().lower()
    if s in ["phish", "phishing", "malicious", "bad", "1", "true"]:
        return "phish"
    if s in ["benign", "legit", "legitimate", "good", "0", "false", "-1"]:
        return "benign"
    return None

def clean_url(u):

    if pd.isna(u):
        return None
    u = str(u).strip().strip('"\'')

    u = re.sub(r"^HTTP://", "http://", u)
    u = re.sub(r"^HTTPS://", "https://", u)

    if not (u.startswith("http://") or u.startswith("https://")):
        return None
    return u if u else None

def basic_clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df.columns = [str(c).strip().lower() for c in df.columns]
    df = df[["sha256", "url", "html", "label"]].copy()

    df = df.dropna(subset=["sha256", "url", "html", "label"])

    df["label"] = df["label"].apply(normalize_label)
    df = df.dropna(subset=["label"])
    df = df[df["label"].isin(["phish", "benign"])]

    df["url"] = df["url"].apply(clean_url)
    df = df.dropna(subset=["url"])

    df["html"] = df["html"].astype(str)
    df["html_len"] = df["html"].str.len()
    df = df[df["html_len"] >= HTML_MIN_LEN]

    df = df.drop_duplicates(subset=["sha256"])
    df = df.drop_duplicates(subset=["url"])

    return df

df0_c = basic_clean(df0)
df3_c = basic_clean(df3)
merged = pd.concat([df0_c, df3_c], ignore_index=True)

merged = merged.drop_duplicates(subset=["sha256"])
merged = merged.drop_duplicates(subset=["url"])

print("After cleaning merged shape:", merged.shape)
print("Label counts:", merged["label"].value_counts(dropna=False))

phish = merged[merged["label"] == "phish"]
benign = merged[merged["label"] == "benign"]

print("Available phish:", len(phish))
print("Available benign:", len(benign))

if len(phish) < N_PER_CLASS or len(benign) < N_PER_CLASS:
    raise ValueError(f"Not enough samples for {N_PER_CLASS}/{N_PER_CLASS}")

balanced = pd.concat(
    [
        phish.sample(n=N_PER_CLASS, random_state=RANDOM_STATE),
        benign.sample(n=N_PER_CLASS, random_state=RANDOM_STATE),
    ],
    ignore_index=True
).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("Balanced shape:", balanced.shape)
print("Balanced label counts:", balanced["label"].value_counts())

out_normal = OUT_DIR / f"balanced_{N_PER_CLASS}_{N_PER_CLASS}_000_003.csv"
out_excel  = OUT_DIR / f"balanced_{N_PER_CLASS}_{N_PER_CLASS}_000_003_excel_safe.csv"

balanced.drop(columns=["html_len"], errors="ignore").to_csv(out_normal, index=False, encoding="utf-8")

balanced.drop(columns=["html_len"], errors="ignore").to_csv(
    out_excel,
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_ALL,
    escapechar="\\"
)

print("Saved:", out_normal)
print("Saved Excel-safe:", out_excel)


After cleaning merged shape: (5554, 5)
Label counts: label
benign    4488
phish     1066
Name: count, dtype: int64
Available phish: 1066
Available benign: 4488
Balanced shape: (2000, 5)
Balanced label counts: label
benign    1000
phish     1000
Name: count, dtype: int64
Saved: c:\Users\14176\Desktop\Honours Project\Hons Project\data\processed\balanced_1000_1000_000_003.csv
Saved Excel-safe: c:\Users\14176\Desktop\Honours Project\Hons Project\data\processed\balanced_1000_1000_000_003_excel_safe.csv


In [3]:
import pandas as pd

p = r"C:\Users\14176\Desktop\Honours Project\Hons Project\data\processed\balanced_1000_1000_000_003.csv"
df = pd.read_csv(p)

print("shape:", df.shape)
print("columns:", df.columns.tolist())
print("label counts:\n", df["label"].value_counts())
print("\nmissing rate:\n", df.isna().mean().sort_values(ascending=False).head(8))

print("\nunique sha256:", df["sha256"].nunique(), " / rows:", len(df))
print("unique url:", df["url"].nunique(), " / rows:", len(df))
print("min html length:", df["html"].astype(str).str.len().min())


shape: (2000, 4)
columns: ['sha256', 'url', 'html', 'label']
label counts:
 label
benign    1000
phish     1000
Name: count, dtype: int64

missing rate:
 sha256    0.0
url       0.0
html      0.0
label     0.0
dtype: float64

unique sha256: 2000  / rows: 2000
unique url: 2000  / rows: 2000
min html length: 1014
